In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
## Load the IMDB dataset

max_features = 10000 ##vocabulary size
(x_train, y_train),(x_test,y_test) = imdb.load_data(num_words=max_features)

# print the shape of the data

print(f'Traing data shape : {x_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {x_test.shape}, Testing labels shape: {y_test.shape}')

Traing data shape : (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [3]:
## inspect sample review

sample_review = x_train[0]
sample_label = y_train[0]

print(f'Sample review (as integer): {sample_review}')
print(f'sample label: {sample_label}')

Sample review (as integer): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
sample label: 1


In [4]:
from tensorflow.keras.preprocessing import sequence

max_len = 500

x_train = sequence.pad_sequences(x_train,maxlen= max_len)
x_test = sequence.pad_sequences(x_test,maxlen= max_len)
x_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]])

In [5]:
x_train[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [8]:
## Train simple RNN

model = Sequential()
model.add(Embedding(max_features, 128, input_length = max_len)) ## Embeddigs Layers
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [9]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy',metrics=['accuracy'])

In [14]:
## create an instance of Earlystopping callback

from tensorflow.keras.callbacks import EarlyStopping
earlystopping = EarlyStopping(monitor = 'val_loss',patience=9, restore_best_weights = True)
earlystopping

In [15]:
## Train the model with early Stopping

model.fit(
    x_train,y_train, epochs = 10, batch_size=32,
    validation_split = 0.2,
    callbacks = [earlystopping]
)

Epoch 1/10


625/625 [==============================] - 94s 144ms/step - loss: 267270064.0000 - accuracy: 0.6048 - val_loss: 0.6371 - val_accuracy: 0.5828
Epoch 2/10
625/625 [==============================] - 81s 129ms/step - loss: 0.6057 - accuracy: 0.6788 - val_loss: 0.6199 - val_accuracy: 0.6266
Epoch 3/10
625/625 [==============================] - 82s 130ms/step - loss: 0.5663 - accuracy: 0.7265 - val_loss: 0.6168 - val_accuracy: 0.6390
Epoch 4/10
625/625 [==============================] - 84s 135ms/step - loss: 0.5068 - accuracy: 0.7598 - val_loss: 0.5490 - val_accuracy: 0.7218
Epoch 5/10
625/625 [==============================] - 85s 136ms/step - loss: 0.3782 - accuracy: 0.8428 - val_loss: 0.4619 - val_accuracy: 0.7982
Epoch 6/10
625/625 [==============================] - 88s 140ms/step - loss: 0.2789 - accuracy: 0.8954 - val_loss: 0.4737 - val_accuracy: 0.7992
Epoch 7/10
625/625 [==============================] - 88s 140ms/step - loss: 0.2033 - accuracy: 0.9276 - val_loss: 0.490

In [16]:
## save model file

model.save('simple_rnn_imdb.h5')

c:\Users\suraj\miniconda3\envs\myenv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
